# 📊 성능 요약 — `4_output/` optuna DB + unit CSV 집계 (pphp 포함)

`4_output/` 아래 모든 실험 폴더를 스캔해서 한눈에 보는 성능표를 만든다. `performance_summary.xlsx`와 같은 컨셉이고, **pphp 노트북(`*_pphp` → `.../{model}/pphp/`)까지 자동으로 잡힌다.**

**소스**: 각 실험 폴더의 `optuna_*.db`(trial 수 / HPO best objective) + `oof|val|test_unit.csv`(실측 unit RMSE — 후처리 적용본, segment 분해 포함).

**시트**:
1. `per_experiment` — 실험 1개 = 1행. trial 진행도 + OOF/val/test RMSE(+ Y=0 / Y>0 분해). `is_pphp` 플래그.
2. `pphp_progress` — pphp 실험만 따로. 매칭되는 hp-only 베이스라인(같은 model의 best 버전)과 trial 수·best RMSE 비교.
3. `two_stage_combined` — `03_two_stage/default/combined/{clf}_x_{reg}/` 페어 (mean / position-weighted).
4. `stacking` — `04_stacking/`의 기존 산출 CSV(base / curated / subset_search) 패스스루.

**출력**: `3_modeling/_check/performance_summary.xlsx` (이 노트북이 덮어씀). 화면에도 출력.

> pphp 실험은 아직 거의 안 돌아서(또는 1 trial RUNNING) RMSE 열이 비어 있을 수 있다 — 그건 정상. 더 돌고 산출물(`*_unit.csv`)이 생기면 이 노트북을 다시 실행하면 채워진다.

## 0. 환경 설정 — 로컬이면 그대로, Colab이면 `4_output.zip` 다운

In [1]:
import os, sys

# Colab에서 4_output(산출물) + utils를 받아 풀기 위한 Drive ID (로컬은 무시)
GDRIVE_CODE_ID   = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py + utils/
GDRIVE_OUTPUT_ID = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'  # 4_output.zip = 전체 산출물 + optuna db

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown openpyxl')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
    if not os.path.exists('/content/project/4_output'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        if os.path.exists('/content/4_output.zip'):
            os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import glob, re, json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from utils.config import PROJECT_ROOT, OUTPUT_DIR, KEY_COL, TARGET_COL

pd.set_option('display.width', 240)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:.6f}' if isinstance(v, float) else str(v))

OUT_XLSX = os.path.join(PROJECT_ROOT, '3_modeling', '_check', 'performance_summary.xlsx')
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'OUTPUT_DIR   = {OUTPUT_DIR}')
print(f'optuna v{optuna.__version__}')
print(f'→ 저장 대상: {OUT_XLSX}')

setup 완료
PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
OUTPUT_DIR   = C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output
optuna v4.7.0
→ 저장 대상: C:\Users\Dell5371\Desktop\기업연계프로젝트\3_modeling\_check\performance_summary.xlsx


## 1. 헬퍼 — RMSE / segment 분해 / optuna DB 읽기 / 경로 파싱

In [2]:
def _rmse(pred, true):
    """NaN 제외하고 RMSE. 둘 다 1D array-like. 유효 샘플 0이면 None."""
    p = np.asarray(pred, dtype=float); t = np.asarray(true, dtype=float)
    m = ~(np.isnan(p) | np.isnan(t))
    if m.sum() == 0:
        return None
    return float(np.sqrt(np.mean((p[m] - t[m]) ** 2)))


def _seg(df):
    """unit CSV(`ufs_serial,pred,health,...`) → 전체/Y=0/Y>0 RMSE + 샘플 수."""
    if df is None or 'pred' not in df.columns or TARGET_COL not in df.columns:
        return dict(rmse=None, rmse_y0=None, rmse_ypos=None, n=0, n_y0=0, n_ypos=0)
    p = df['pred'].values; t = df[TARGET_COL].values
    y0 = t == 0; yp = t > 0
    return dict(
        rmse=_rmse(p, t),
        rmse_y0=_rmse(p[y0], t[y0]) if y0.any() else None,
        rmse_ypos=_rmse(p[yp], t[yp]) if yp.any() else None,
        n=int(len(t)), n_y0=int(y0.sum()), n_ypos=int(yp.sum()),
    )


def _read_unit(path):
    return pd.read_csv(path) if os.path.exists(path) else None


def _db_info(exp_dir):
    """실험 폴더의 optuna*.db에서 trial 통계 + best objective + study user_attrs 일부. 없으면 None."""
    dbs = sorted(glob.glob(os.path.join(exp_dir, 'optuna*.db')))
    if not dbs:
        return None
    best = None
    for db in dbs:
        storage = 'sqlite:///' + db.replace(os.sep, '/')
        try:
            names = optuna.get_all_study_names(storage)
        except Exception:
            continue
        for nm in names:
            try:
                st = optuna.load_study(study_name=nm, storage=storage)
            except Exception:
                continue
            tr = st.trials
            info = dict(
                db=os.path.relpath(db, OUTPUT_DIR).replace(os.sep, '/'),
                study_name=nm,
                n_total=len(tr),
                n_complete=sum(1 for t in tr if t.state.name == 'COMPLETE'),
                n_pruned=sum(1 for t in tr if t.state.name == 'PRUNED'),
                n_running=sum(1 for t in tr if t.state.name == 'RUNNING'),
                n_fail=sum(1 for t in tr if t.state.name == 'FAIL'),
            )
            try:
                info['best_value'] = float(st.best_value)
                info['best_trial'] = int(st.best_trial.number)
            except Exception:
                info['best_value'] = None; info['best_trial'] = None
            ua = dict(st.user_attrs)
            info['exp_memo']   = ua.get('exp_memo') or ua.get('exp_id')
            info['model_meta'] = ua.get('model_name') or ua.get('reg_model_name') or ua.get('clf_model_name')
            info['is_pphp_meta'] = bool(ua.get('pp_search_candidates') is not None or ua.get('pp_cache_size') is not None)
            # 한 폴더에 study가 여러 개면 trial 가장 많은 걸 대표로
            if best is None or info['n_total'] > best['n_total']:
                best = info
    return best


def _parse_path(rel):
    """OUTPUT_DIR 기준 상대경로 → (section, model, version, kind, is_pphp)."""
    parts = rel.split('/')
    is_pphp = ('pphp' in parts)
    if parts[0] == '01_zit':
        return '01_zit', parts[1], parts[-1], 'zit', is_pphp
    if parts[0] == '02_reg_single':
        return '02_reg_single', parts[1], parts[-1], 'reg_single', is_pphp
    if parts[0] == '03_two_stage':
        if len(parts) >= 4 and parts[1] == 'default' and parts[2] == 'clf':
            return '03_two_stage/clf', parts[3], parts[-1], 'two_stage_clf', is_pphp
        if len(parts) >= 4 and parts[1] == 'default' and parts[2] == 'reg':
            return '03_two_stage/reg', parts[3], parts[-1], 'two_stage_reg', is_pphp
        if len(parts) >= 2 and parts[1] == 'reverse':
            return '03_two_stage/reverse', 'ts_reverse', parts[-1], 'two_stage_reverse', is_pphp
    if parts[0] == 'baseline':
        return 'baseline', 'e2e', parts[-1], 'baseline_e2e', is_pphp
    return parts[0], (parts[1] if len(parts) > 1 else '?'), parts[-1], 'other', is_pphp


def _discover_exp_dirs():
    """`4_output/` 아래 '실험 폴더'(optuna db 또는 oof/val/test_unit.csv 보유)를 찾는다.
    04_stacking/ 와 03_two_stage/.../combined/ 는 따로 다루므로 제외."""
    dirs = []
    for dp, dn, fn in os.walk(OUTPUT_DIR):
        rel = os.path.relpath(dp, OUTPUT_DIR).replace(os.sep, '/')
        if rel == '.':
            continue
        if rel.startswith('04_stacking'):
            continue
        if re.search(r'(^|/)combined($|/)', rel):   # combined 페어 dir + 그 부모 skip
            continue
        has_db   = any(f.startswith('optuna') and f.endswith('.db') for f in fn)
        has_unit = any(f in ('oof_unit.csv', 'val_unit.csv', 'test_unit.csv') for f in fn)
        if has_db or has_unit:
            dirs.append(dp)
    return sorted(set(dirs))


print('헬퍼 정의 완료')

헬퍼 정의 완료


## 2. `per_experiment` — 실험별 trial 진행도 + 실측 RMSE

- `hpo_best_objective` = optuna study의 best value(= train OOF unit RMSE objective, HPO 중 fold 분할 기준).
- `oof_rmse / val_rmse / test_rmse` = `oof|val|test_unit.csv`(refit + 후처리 적용본)에서 직접 계산. `*_y0`/`*_ypos`는 health==0 / health>0 segment.
- pphp 행은 `version=='pphp'` + `is_pphp=True`. 아직 산출물 없으면 RMSE 열은 NaN, trial 열만 채워짐.

In [3]:
rows = []
for d in _discover_exp_dirs():
    rel = os.path.relpath(d, OUTPUT_DIR).replace(os.sep, '/')
    section, model, version, kind, is_pphp = _parse_path(rel)
    dbi = _db_info(d) or {}
    is_pphp = is_pphp or bool(dbi.get('is_pphp_meta'))
    oof = _seg(_read_unit(os.path.join(d, 'oof_unit.csv')))
    val = _seg(_read_unit(os.path.join(d, 'val_unit.csv')))
    tst = _seg(_read_unit(os.path.join(d, 'test_unit.csv')))
    rows.append(dict(
        section=section, model=model, version=version, kind=kind, is_pphp=is_pphp,
        rel_path='4_output/' + rel,
        n_trials_total=dbi.get('n_total'), n_trials_complete=dbi.get('n_complete'),
        n_trials_pruned=dbi.get('n_pruned'), n_trials_running=dbi.get('n_running'),
        hpo_best_objective=dbi.get('best_value'),
        oof_rmse=oof['rmse'], val_rmse=val['rmse'], test_rmse=tst['rmse'],
        oof_rmse_y0=oof['rmse_y0'], val_rmse_y0=val['rmse_y0'], test_rmse_y0=tst['rmse_y0'],
        oof_rmse_ypos=oof['rmse_ypos'], val_rmse_ypos=val['rmse_ypos'], test_rmse_ypos=tst['rmse_ypos'],
        n_oof=oof['n'], n_val=val['n'], n_test=tst['n'],
        exp_memo=dbi.get('exp_memo'),
    ))

per_experiment = pd.DataFrame(rows)
# 정렬: kind → oof_rmse 오름차순(없으면 맨 아래)
_kind_order = {'reg_single': 0, 'two_stage_clf': 1, 'two_stage_reg': 2, 'two_stage_reverse': 3, 'zit': 4, 'baseline_e2e': 5, 'other': 9}
per_experiment = (per_experiment
    .assign(_k=per_experiment['kind'].map(_kind_order).fillna(8),
            _r=per_experiment['oof_rmse'].fillna(np.inf))
    .sort_values(['_k', '_r', 'val_rmse'])
    .drop(columns=['_k', '_r'])
    .reset_index(drop=True))

_show = ['section','model','version','is_pphp','n_trials_total','n_trials_complete','n_trials_running',
         'hpo_best_objective','oof_rmse','val_rmse','test_rmse','oof_rmse_y0','oof_rmse_ypos']
print(f'총 {len(per_experiment)}개 실험 폴더 (pphp {int(per_experiment.is_pphp.sum())}개)')
per_experiment[_show]

총 38개 실험 폴더 (pphp 3개)


,section,model,version,is_pphp,n_trials_total,n_trials_complete,n_trials_running,hpo_best_objective,oof_rmse,val_rmse,test_rmse,oof_rmse_y0,oof_rmse_ypos
0,02_reg_single,lgbm,002,False,217.000000,217.000000,0.000000,0.005512,0.005509,0.005722,0.008428,0.002374,0.009502
1,02_reg_single,lgbm,001,False,57.000000,57.000000,0.000000,0.005518,0.005518,0.005726,0.008427,0.002474,0.009457
2,02_reg_single,et,002,False,30.000000,30.000000,0.000000,0.005520,0.005520,0.005734,0.008433,0.002506,0.009440
3,02_reg_single,lgbm,median_impute,False,NaN,NaN,NaN,NaN,0.005520,0.005729,0.008428,0.002478,0.009459
4,02_reg_single,lgbm,knn_impute,False,NaN,NaN,NaN,NaN,0.005520,0.005728,0.008428,0.002477,0.009460
5,02_reg_single,lgbm,knn_scaled_impute,False,NaN,NaN,NaN,NaN,0.005520,0.005729,0.008427,0.002478,0.009459
6,02_reg_single,catboost,001,False,26.000000,26.000000,0.000000,0.005523,0.005522,0.005730,0.008429,0.002506,0.009445
7,02_reg_single,catboost,002,False,44.000000,44.000000,0.000000,0.005523,0.005523,0.005733,0.008432,0.002491,0.009456
8,02_reg_single,xgb,001,False,49.000000,49.000000,0.000000,0.005525,0.005526,0.005728,0.008423,0.002486,0.009466
9,02_reg_single,et,001,False,59.000000,59.000000,0.000000,0.005546,0.005540,0.005755,0.008450,0.002473,0.009501


## 3. `pphp_progress` — pphp 실험 진행 현황 (hp-only 베이스라인 대비)

각 pphp 실험에 대해, **같은 `kind`·`model`의 hp-only 실험 중 oof_rmse가 가장 좋은 것**을 베이스라인으로 잡아 trial 수·best objective를 나란히 본다. pphp는 trial마다 전처리(spatial impute 포함)를 다시 도므로 hp-only보다 throughput이 낮다 — 그래서 `n_trials_total`이 작은 게 정상.

In [4]:
pphp = per_experiment[per_experiment.is_pphp].copy()
base = per_experiment[~per_experiment.is_pphp].copy()

rows = []
for _, r in pphp.iterrows():
    cand = base[(base.kind == r.kind) & (base.model == r.model)].copy()
    cand = cand.sort_values('oof_rmse')   # oof_rmse 좋은 순 (NaN은 맨 뒤)
    b = cand.iloc[0] if len(cand) else None
    rows.append(dict(
        kind=r.kind, model=r.model, pphp_path=r.rel_path,
        pphp_trials_total=r.n_trials_total, pphp_trials_complete=r.n_trials_complete,
        pphp_trials_running=r.n_trials_running, pphp_trials_pruned=r.n_trials_pruned,
        pphp_hpo_best=r.hpo_best_objective, pphp_oof_rmse=r.oof_rmse, pphp_val_rmse=r.val_rmse,
        baseline_version=(b.version if b is not None else None),
        baseline_path=(b.rel_path if b is not None else None),
        baseline_trials_total=(b.n_trials_total if b is not None else None),
        baseline_hpo_best=(b.hpo_best_objective if b is not None else None),
        baseline_oof_rmse=(b.oof_rmse if b is not None else None),
        baseline_val_rmse=(b.val_rmse if b is not None else None),
        delta_hpo_best=((r.hpo_best_objective - b.hpo_best_objective)
                        if (b is not None and r.hpo_best_objective is not None and b.hpo_best_objective is not None) else None),
    ))
pphp_progress = pd.DataFrame(rows)
if len(pphp_progress):
    pphp_progress = pphp_progress.sort_values(['kind', 'model']).reset_index(drop=True)
    n_done = int((pphp_progress.pphp_trials_complete.fillna(0) > 0).sum())
    print(f'pphp 실험 {len(pphp_progress)}개 — complete trial 1개 이상인 것: {n_done}개')
    print('(complete=0 이면 아직 안 돌았거나 trial 0이 RUNNING 중 — Colab에서 더 돌리고 4_output.zip 갱신하면 채워짐)')
else:
    print('pphp 실험 폴더 없음 (아직 한 번도 안 돌림 — 정상)')
pphp_progress

pphp 실험 3개 — complete trial 1개 이상인 것: 1개
(complete=0 이면 아직 안 돌았거나 trial 0이 RUNNING 중 — Colab에서 더 돌리고 4_output.zip 갱신하면 채워짐)


,kind,model,pphp_path,pphp_trials_total,pphp_trials_complete,pphp_trials_running,pphp_trials_pruned,pphp_hpo_best,pphp_oof_rmse,pphp_val_rmse,baseline_version,baseline_path,baseline_trials_total,baseline_hpo_best,baseline_oof_rmse,baseline_val_rmse,delta_hpo_best
0,reg_single,catboost,4_output/02_reg_single/catboost/pphp,1.000000,0.000000,1.000000,0.000000,NaN,NaN,NaN,001,4_output/02_reg_single/catboost/001,26.000000,0.005523,0.005522,0.005730,NaN
1,reg_single,lgbm,4_output/02_reg_single/lgbm/pphp,1.000000,1.000000,0.000000,0.000000,0.005522,NaN,NaN,002,4_output/02_reg_single/lgbm/002,217.000000,0.005512,0.005509,0.005722,0.000011
2,reg_single,xgb,4_output/02_reg_single/xgb/pphp,1.000000,0.000000,1.000000,0.000000,NaN,NaN,NaN,001,4_output/02_reg_single/xgb/001,49.000000,0.005525,0.005526,0.005728,NaN


## 4. `two_stage_combined` — Stage1(clf) × Stage2(reg) 페어 grid

`03_two_stage/default/combined/{clf}_x_{reg}/` 의 `oof|val|test_unit.csv`(mean 집계) + `*_unit_weighted.csv`(position 가중평균). pphp 산출물이 나중에 이 grid에 추가되면 (`combine.ipynb`가 입력 풀에 `pphp/`를 넣으면) 여기서 자동으로 같이 잡힌다.

In [5]:
cdir = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'combined')
rows = []
for pdir in sorted(glob.glob(os.path.join(cdir, '*_x_*'))):
    if not os.path.isdir(pdir):
        continue
    m = re.match(r'(.+?)_x_(.+)', os.path.basename(pdir))
    if not m:
        continue
    clf, reg = m.group(1), m.group(2)
    for variant, suffix in [('mean', ''), ('weighted', '_weighted')]:
        o = _seg(_read_unit(os.path.join(pdir, f'oof_unit{suffix}.csv')))
        v = _seg(_read_unit(os.path.join(pdir, f'val_unit{suffix}.csv')))
        t = _seg(_read_unit(os.path.join(pdir, f'test_unit{suffix}.csv')))
        if o['rmse'] is None and v['rmse'] is None and t['rmse'] is None:
            continue
        rows.append(dict(
            clf=clf, reg=reg, variant=variant,
            oof_rmse=o['rmse'], val_rmse=v['rmse'], test_rmse=t['rmse'],
            oof_rmse_y0=o['rmse_y0'], oof_rmse_ypos=o['rmse_ypos'],
            val_rmse_y0=v['rmse_y0'], val_rmse_ypos=v['rmse_ypos'],
            test_rmse_y0=t['rmse_y0'], test_rmse_ypos=t['rmse_ypos'],
            rel_path=os.path.relpath(pdir, PROJECT_ROOT).replace(os.sep, '/'),
        ))
two_stage_combined = pd.DataFrame(rows)
if len(two_stage_combined):
    two_stage_combined = two_stage_combined.sort_values(['val_rmse', 'oof_rmse']).reset_index(drop=True)
    print(f'{len(two_stage_combined)}개 (clf×reg×variant) 조합')
else:
    print('combined 페어 산출물 없음')
two_stage_combined.head(20)

40개 (clf×reg×variant) 조합


,clf,reg,variant,oof_rmse,val_rmse,test_rmse,oof_rmse_y0,oof_rmse_ypos,val_rmse_y0,val_rmse_ypos,test_rmse_y0,test_rmse_ypos,rel_path
0,lgbm,et,mean,0.008243,0.005709,0.008413,0.002493,0.014754,0.002488,0.009828,0.002477,0.015098,4_output/03_two_stage/default/combined/lgbm_x_et
1,xgb,et,mean,0.008242,0.005710,0.008414,0.002486,0.014755,0.002482,0.009833,0.002469,0.015102,4_output/03_two_stage/default/combined/xgb_x_et
2,lgbm,et,weighted,0.008244,0.005711,0.008414,0.002506,0.014749,0.002501,0.009825,0.002489,0.015094,4_output/03_two_stage/default/combined/lgbm_x_et
3,xgb,et,weighted,0.008243,0.005712,0.008414,0.002498,0.014751,0.002494,0.009830,0.002481,0.015098,4_output/03_two_stage/default/combined/xgb_x_et
4,catboost,et,mean,0.008246,0.005714,0.008416,0.002473,0.014767,0.002477,0.009845,0.002459,0.015111,4_output/03_two_stage/default/combined/catboos...
5,catboost,et,weighted,0.008247,0.005718,0.008418,0.002504,0.014755,0.002506,0.009835,0.002489,0.015102,4_output/03_two_stage/default/combined/catboos...
6,et,et,mean,0.008246,0.005721,0.008422,0.002522,0.014746,0.002528,0.009827,0.002516,0.015100,4_output/03_two_stage/default/combined/et_x_et
7,et,et,weighted,0.008247,0.005725,0.008424,0.002546,0.014739,0.002551,0.009821,0.002539,0.015094,4_output/03_two_stage/default/combined/et_x_et
8,lgbm,catboost,mean,0.008284,0.005756,0.008456,0.001956,0.015025,0.001936,0.010216,0.001930,0.015372,4_output/03_two_stage/default/combined/lgbm_x_...
9,lgbm,enet,mean,0.008276,0.005756,0.008449,0.001936,0.015016,0.001934,0.010217,0.001923,0.015360,4_output/03_two_stage/default/combined/lgbm_x_...


## 5. `stacking` — `04_stacking/` 기존 산출물 패스스루

stacking은 별도 노트북(`04_stacking/stacking.ipynb`)에서 만든 산출물이 이미 CSV/JSON으로 있어서, 여기서는 그대로 읽어와 시트로만 옮긴다. (pphp OOF 15벌이 base pool에 추가되는 건 `stacking.ipynb` 쪽 후속 작업 — 그게 끝나면 이 시트도 자동 갱신.)

In [6]:
def _try_csv(rel, **kw):
    p = os.path.join(OUTPUT_DIR, rel)
    if not os.path.exists(p):
        return None
    try:
        return pd.read_csv(p, **kw)
    except Exception as e:
        print(f'  [skip] {rel}: {e}')
        return None

stacking_base_comparison   = _try_csv('04_stacking/base/comparison.csv')
stacking_base_single       = _try_csv('04_stacking/base/single_base_rmse.csv')
stacking_curated_configs   = _try_csv('04_stacking/curated/config_comparison.csv')
stacking_curated_candidate = _try_csv('04_stacking/curated/candidate_single_rmse.csv')
_subset = _try_csv('04_stacking/_subset_search/results.csv')
stacking_subset_top = (_subset.sort_values('val').head(25).reset_index(drop=True)
                       if (_subset is not None and 'val' in _subset.columns) else _subset)

# curated winner 의 enet 계수 (meta.json) → DataFrame
stacking_winner_coef = None
_cm = os.path.join(OUTPUT_DIR, '04_stacking', 'curated', 'meta.json')
if os.path.exists(_cm):
    try:
        _w = json.load(open(_cm, encoding='utf-8')).get('winner', {})
        _coef = _w.get('coef', {})
        _sng = {}
        if stacking_curated_candidate is not None:
            _sng = stacking_curated_candidate.set_index('model')[['val', 'test']].to_dict('index')
        stacking_winner_coef = pd.DataFrame([
            dict(base_model=k, enet_coef=v,
                 single_val=_sng.get(k, {}).get('val'), single_test=_sng.get(k, {}).get('test'))
            for k, v in sorted(_coef.items(), key=lambda kv: -abs(kv[1]))
        ])
    except Exception as e:
        print(f'  [skip] curated/meta.json: {e}')

for nm, df in [('base/comparison', stacking_base_comparison), ('base/single', stacking_base_single),
               ('curated/configs', stacking_curated_configs), ('curated/candidates', stacking_curated_candidate),
               ('subset_search top25', stacking_subset_top), ('curated winner coef', stacking_winner_coef)]:
    print(f'  {nm:24s} → {"None" if df is None else str(df.shape)}')

stacking_curated_configs

  base/comparison          → (5, 4)
  base/single              → (24, 4)
  curated/configs          → (9, 10)
  curated/candidates       → (15, 4)
  subset_search top25      → (25, 15)
  curated winner coef      → (10, 4)


,name,cv_rmse,oof,val,test,n_base,n_active,alpha,l1_ratio,cols
0,forward_selection(10),0.005490,0.005489,0.005701,0.008408,10,10,0.000001,0.100000,zit_only|old_bagzit_fixed_ge|reg_xgb|ts_revers...
1,all 15 candidates,0.005490,0.005488,0.005701,0.008408,15,11,0.000004,0.700000,zit_only|bag_zit|ts_reverse|reg_lgbm|reg_xgb|r...
2,old-pool reconstruct (zit+5oldbag+reg4),0.005491,0.005490,0.005701,0.008407,10,9,0.000004,1.000000,zit_only|old_bagzit_combined|old_bagzit_combin...
3,zit3 + 5 old bagzit variants,0.005492,0.005491,0.005702,0.008408,8,6,0.000004,0.900000,zit_only|bag_zit|ts_reverse|old_bagzit_combine...
4,zit3 + reg4(new) + grid3,0.005494,0.005492,0.005704,0.008410,10,9,0.000003,0.900000,zit_only|bag_zit|ts_reverse|reg_lgbm|reg_xgb|r...
5,exhaustive_top9_byCV,0.005494,0.005493,0.005704,0.008411,5,5,0.000001,0.100000,zit_only|ts_reverse|grid_lgbm_x_et|old_bagzit_...
6,zit3 [zit+bag+ts_rev],0.005495,0.005494,0.005704,0.008410,3,3,0.000028,0.100000,zit_only|bag_zit|ts_reverse
7,zit3 + reg_lgbm,0.005495,0.005494,0.005704,0.008410,4,3,0.000073,0.100000,zit_only|bag_zit|ts_reverse|reg_lgbm
8,exhaustive_top9_byVAL,0.005495,0.005495,0.005703,0.008409,3,3,0.000017,0.100000,zit_only|bag_zit|old_bagzit_hpo


## 6. `performance_summary.xlsx` 저장

In [7]:
sheets = {
    'per_experiment':      per_experiment,
    'pphp_progress':       pphp_progress,
    'two_stage_combined':  two_stage_combined,
    'stacking_base':       stacking_base_comparison,
    'stacking_base_single':stacking_base_single,
    'stacking_curated':    stacking_curated_configs,
    'stacking_subset_top': stacking_subset_top,
    'stacking_winner_coef':stacking_winner_coef,
}
os.makedirs(os.path.dirname(OUT_XLSX), exist_ok=True)
with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as xw:
    for nm, df in sheets.items():
        if df is None or len(df) == 0:
            pd.DataFrame({'(no data)': []}).to_excel(xw, sheet_name=nm[:31], index=False)
        else:
            df.to_excel(xw, sheet_name=nm[:31], index=False)
print(f'저장 완료 → {OUT_XLSX}')
for nm, df in sheets.items():
    print(f'  [{nm}] {0 if df is None else len(df)} rows')

# Colab이면 다운로드
try:
    import google.colab
    from google.colab import files
    files.download(OUT_XLSX)
except Exception:
    pass

저장 완료 → C:\Users\Dell5371\Desktop\기업연계프로젝트\3_modeling\_check\performance_summary.xlsx
  [per_experiment] 38 rows
  [pphp_progress] 3 rows
  [two_stage_combined] 40 rows
  [stacking_base] 5 rows
  [stacking_base_single] 24 rows
  [stacking_curated] 9 rows
  [stacking_subset_top] 25 rows
  [stacking_winner_coef] 10 rows
